In [ ]:
from modelo import cargar_instancia, crear_modelo_convrp
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
from IPython.display import display
import os

# Modelos

In [ ]:
def optimizar_modelo_base(datos):
    modelo = crear_modelo_convrp(datos)
    
    modelo.Params.Threads = 1
    modelo.Params.TimeLimit = 600
    
    print("¡Optimizando modelo base!\n")
    modelo.optimize()
    return modelo

def optimizar_modelo_relajado(datos):
    modelo = crear_modelo_convrp(datos)
    
    # Convierte las variables binarias a continuas
    modelo = modelo.relax()
    
    modelo.Params.Threads = 1
    modelo.Params.TimeLimit = 600
    
    print("¡Optimizando modelo relajado!\n")
    modelo.optimize()
    return modelo

def optimizar_modelo_cuts(datos):
    modelo = crear_modelo_convrp(datos)
    
    modelo.Params.Threads = 1
    modelo.Params.TimeLimit = 600
    modelo.Params.Cuts = 0
    
    print("¡Optimizando modelo con cuts = 0!\n")
    modelo.optimize()
    return modelo

def optimizar_modelo_presolve(datos):
    modelo = crear_modelo_convrp(datos)
    
    modelo.Params.Threads = 1
    modelo.Params.TimeLimit = 600
    modelo.Params.Presolve = 0
    
    print("¡Optimizando modelo con presolve = 0!\n")
    modelo.optimize()
    return modelo

def optimizar_modelo_mipfocus_1(datos):
    modelo = crear_modelo_convrp(datos)
    
    modelo.Params.Threads = 1
    modelo.Params.TimeLimit = 600
    modelo.Params.MIPFocus = 1
    
    print("¡Optimizando modelo con MIPFocus = 1!\n")
    modelo.optimize()
    return modelo

def optimizar_modelo_mipfocus_3(datos):
    modelo = crear_modelo_convrp(datos)
    
    modelo.Params.Threads = 1
    modelo.Params.TimeLimit = 600
    modelo.Params.MIPFocus = 3
    
    print("¡Optimizando modelo con MIPFocus = 3!\n")
    modelo.optimize()
    return modelo

def optimizar_modelo_heuristics(datos):
    modelo = crear_modelo_convrp(datos)
    
    modelo.Params.Threads = 1
    modelo.Params.TimeLimit = 600
    modelo.Params.Heuristics = 0
    
    print("¡Optimizando modelo con heuristics = 0!\n")
    modelo.optimize()
    return modelo

def optimizar_modelo_tuneado(datos, nombre_instancia):
    modelo = crear_modelo_convrp(datos)
    
    modelo.Params.Threads = 1
    modelo.Params.TimeLimit = 600
    modelo.Params.TuneTimeLimit = 3600
    
    print("¡Optimizando parametros!\n")
    modelo.tune()
    
    if modelo.TuneResultCount > 0:
        modelo.getTuneResult(0)
        os.makedirs("parametros", exist_ok=True)
        modelo.write(f"parametros/mejores_parametros_{nombre_instancia}.prm")
    
    print("¡Optimizando modelo con parametros tuneados!")
    modelo.optimize()
    return modelo

# Funciones para análisis de resultados

In [ ]:
def reporte_modelo(modelo, nombre_modelo, nombre_instancia):
    tiempo_ejecucion = modelo.Runtime
    
    if modelo.SolCount > 0:
        valor_objetivo = modelo.ObjVal
        gap_final = modelo.MIPGap
        
        print(f"\n¡Reporte {nombre_modelo} {nombre_instancia}!\n"
                  f"Tiempo de ejecución: {tiempo_ejecucion:.2f}\n"
                  f"Valor objetivo: {valor_objetivo}\n"
                  f"Gap final: {gap_final}\n")
    else:
        modelo.computeIIS()
        modelo.write("modelo_infactible.ilp")
        print("Archivo 'modelo_infactible.ilp' generado con éxito.")
        
    

def comparacion_baseline_relajado(baseline, relajado):
    if baseline.SolCount > 0 and relajado.SolCount > 0:
        brecha_integralidad = 100 * (baseline.ObjVal - relajado.ObjVal) / baseline.ObjVal
        print(f"Valor objetivo modelo baseline: {baseline.ObjVal:2f}\n"
            f"Valor objetivo modelo relajado: {relajado.ObjVal:2f}\n"
            f"Brecha de integralidad: {brecha_integralidad}%\n")
    else:
        print("No se puede calcular la brecha de integralidad puesto que nos e obtuvo una solucion factible en alguno de los modelos.\n")

def comparacion_modelos(modelos, nombres, instancia):
    data = {
            "Modelo": [],
            "Tiempo": [],
            "Valor Objetivo": [],
            "Gap Final": []
            }
    
    for i, m in enumerate(modelos):
        data["Modelo"].append(nombres[i])
        data["Tiempo"].append(m.Runtime)
        data["Valor Objetivo"].append(m.ObjVal)
        data["Gap Final"].append(m.MIPGap)
    
    print(f"=== Comparativa de modelos {instancia} ===\n")
    df = pd.DataFrame(data)
    display(df)
    
def analisis_sensibilidad_capacidad(modelo, nombre_instancia, tol=1e-6):
    if modelo.SolCount == 0:
        print(f"[{nombre_instancia}] No hay solución incumbente: no se puede fijar el modelo.")
        return None, None

    fm = modelo.fixed()          # MIP -> LP con enteras fijas en la incumbente
    fm.Params.Presolve = 0       # evita que se eliminen filas con solo variables fijas
    fm.Params.Method = 1         # simplex dual
    fm.Params.Threads = 1
    fm.Params.OutputFlag = 0
    fm.optimize()

    if fm.Status != GRB.OPTIMAL:
        print(f"[{nombre_instancia}] El modelo fijado no es óptimo (estado {fm.Status}).")
        return None, fm

    filas = []
    for c in fm.getConstrs():
        # Criterio 1: Nombre explícito que empiece con "Capacidad"
        es_capacidad = c.ConstrName.lower().startswith("capacidad")
        
        # Criterio 2: Restricción de tipo <= compuesta por variables Z
        if not es_capacidad and c.Sense == '<':
            row = fm.getRow(c)
            if row.size() > 0:
                es_capacidad = all(row.getVar(i).VarName.startswith("Z_") for i in range(row.size()))

        if es_capacidad:
            carga = c.RHS - c.Slack
            nombre_mostrar = c.ConstrName if c.ConstrName.lower().startswith("capacidad") else f"Capacidad ({c.ConstrName})"
            filas.append({
                "Restricción": nombre_mostrar,
                "Q (RHS)": c.RHS,
                "Carga": carga,
                "Holgura": c.Slack,
                "Activa": c.Slack < tol,
                "Precio sombra (Pi)": c.Pi,
                "SARHSLow": c.SARHSLow,
                "SARHSUp": c.SARHSUp,
            })

    df = pd.DataFrame(filas)

    # Validación de seguridad si no se encuentran filas
    if df.empty:
        print(f"=== Sensibilidad de Q, instancia {nombre_instancia} ===")
        print(f"[{nombre_instancia}] No se identificaron restricciones de capacidad en el modelo.")
        return None, fm

    print(f"=== Sensibilidad de Q, instancia {nombre_instancia} ===")
    print(f"Objetivo MIP: {modelo.ObjVal:.4f} | Objetivo LP fijado: {fm.ObjVal:.4f}")
    print(f"Restricciones de capacidad: {len(df)} | activas: {int(df['Activa'].sum())}")
    print(f"Carga máxima observada: {df['Carga'].max():.2f} (Q = {df['Q (RHS)'].iloc[0]:.2f})")
    print(f"Filas con precio sombra distinto de 0: {(df['Precio sombra (Pi)'].abs() > tol).sum()}")
    display(df.sort_values("Carga", ascending=False).head(15))
    
    return df, fm

In [4]:
datos_1 = cargar_instancia("instancias/C108.json")
datos_2 = cargar_instancia("instancias/R108.json")

## Modelo Base

In [5]:
modelo_base_I1 = optimizar_modelo_base(datos_1)
reporte_modelo(modelo_base_I1, "modelo base", "I1")

Set parameter Username
Set parameter LicenseID to value 2875196
Academic license - for non-commercial use only - expires 2027-10-15
Set parameter Threads to value 1
Set parameter TimeLimit to value 600
¡Optimizando modelo base!

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-11800H @ 2.30GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 1 threads

Non-default parameters:
TimeLimit  600
Threads  1

Optimize a model with 1428 rows, 1287 columns and 6300 nonzeros (Min)
Model fingerprint: 0xcbc56733
Model has 1188 linear objective coefficients
Variable types: 45 continuous, 1242 integer (1242 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+03]
  Objective range  [1e+00, 4e+01]
  Bounds range     [1e+00, 1e+03]
  RHS range        [1e+00, 1e+03]

Presolve removed 783 rows and 397 columns
Presolve time: 0.01s
Presolved: 645 rows, 890 columns,

In [6]:
modelo_base_I2 = optimizar_modelo_base(datos_2)
reporte_modelo(modelo_base_I2, "modelo base", "I2")

Set parameter Threads to value 1
Set parameter TimeLimit to value 600
¡Optimizando modelo base!

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-11800H @ 2.30GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 1 threads

Non-default parameters:
TimeLimit  600
Threads  1

Optimize a model with 1506 rows, 1360 columns and 6669 nonzeros (Min)
Model fingerprint: 0xc86c9a53
Model has 1260 linear objective coefficients
Variable types: 46 continuous, 1314 integer (1314 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+02]
  Objective range  [7e+00, 6e+01]
  Bounds range     [1e+00, 2e+02]
  RHS range        [1e+00, 2e+02]

Presolve removed 312 rows and 147 columns
Presolve time: 0.03s
Presolved: 1194 rows, 1213 columns, 5331 nonzeros
Variable types: 34 continuous, 1179 integer (1179 binary)

Root relaxation: objective 5.394012e+02, 540 iterations,

## Modelo Relajado

In [7]:
modelo_relajado_I1 = optimizar_modelo_relajado(datos_1)

# Comparacion del baseline con el relajado
comparacion_baseline_relajado(modelo_base_I1, modelo_relajado_I1)

Set parameter Threads to value 1
Set parameter TimeLimit to value 600
¡Optimizando modelo relajado!

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-11800H @ 2.30GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 1 threads

Non-default parameters:
TimeLimit  600
Threads  1

Optimize a model with 1428 rows, 1287 columns and 6300 nonzeros (Min)
Model fingerprint: 0xbdbb084e
Model has 1188 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+03]
  Objective range  [1e+00, 4e+01]
  Bounds range     [1e+00, 1e+03]
  RHS range        [1e+00, 1e+03]

Presolve removed 405 rows and 27 columns
Presolve time: 0.02s
Presolved: 1023 rows, 1260 columns, 5061 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   3.150000e+01   0.000000e+00      0s
     161    1.2514096e+02   0.000000e+00 

In [8]:
modelo_relajado_I2 = optimizar_modelo_relajado(datos_2)

# Comparacion del baseline con el relajado
comparacion_baseline_relajado(modelo_base_I2, modelo_relajado_I2)

Set parameter Threads to value 1


Set parameter TimeLimit to value 600
¡Optimizando modelo relajado!

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-11800H @ 2.30GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 1 threads

Non-default parameters:
TimeLimit  600
Threads  1

Optimize a model with 1506 rows, 1360 columns and 6669 nonzeros (Min)
Model fingerprint: 0x811a7184
Model has 1260 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 3e+02]
  Objective range  [7e+00, 6e+01]
  Bounds range     [1e+00, 2e+02]
  RHS range        [1e+00, 2e+02]

Presolve removed 192 rows and 27 columns
Presolve time: 0.01s
Presolved: 1314 rows, 1333 columns, 6029 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   3.300000e+01   0.000000e+00      0s
     206    5.2091919e+02   0.000000e+00   0.000000e+00      0s

Use cross

## Prueba de configuración de modelos
### Modelo Cuts = 0

In [9]:
# Optimización del modelo con configuración 1 con ambas instancias
modelo_cuts_I1 = optimizar_modelo_cuts(datos_1)
reporte_modelo(modelo_cuts_I1, "modelo Cuts = 0", "I1")

Set parameter Threads to value 1
Set parameter TimeLimit to value 600
Set parameter Cuts to value 0
¡Optimizando modelo con cuts = 0!

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-11800H @ 2.30GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 1 threads

Non-default parameters:
TimeLimit  600
Cuts  0
Threads  1

Optimize a model with 1428 rows, 1287 columns and 6300 nonzeros (Min)
Model fingerprint: 0xcbc56733
Model has 1188 linear objective coefficients
Variable types: 45 continuous, 1242 integer (1242 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+03]
  Objective range  [1e+00, 4e+01]
  Bounds range     [1e+00, 1e+03]
  RHS range        [1e+00, 1e+03]

Presolve removed 783 rows and 397 columns
Presolve time: 0.02s
Presolved: 645 rows, 890 columns, 3102 nonzeros
Variable types: 32 continuous, 858 integer (858 binary)

Root relaxatio

In [10]:
modelo_cuts_I2 = optimizar_modelo_cuts(datos_2)
reporte_modelo(modelo_cuts_I2, "modelo Cuts = 0", "I2")

Set parameter Threads to value 1
Set parameter TimeLimit to value 600
Set parameter Cuts to value 0
¡Optimizando modelo con cuts = 0!

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-11800H @ 2.30GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 1 threads

Non-default parameters:
TimeLimit  600
Cuts  0
Threads  1

Optimize a model with 1506 rows, 1360 columns and 6669 nonzeros (Min)
Model fingerprint: 0xc86c9a53
Model has 1260 linear objective coefficients
Variable types: 46 continuous, 1314 integer (1314 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+02]
  Objective range  [7e+00, 6e+01]
  Bounds range     [1e+00, 2e+02]
  RHS range        [1e+00, 2e+02]

Presolve removed 312 rows and 147 columns
Presolve time: 0.06s
Presolved: 1194 rows, 1213 columns, 5331 nonzeros
Variable types: 34 continuous, 1179 integer (1179 binary)

Root relax

### Modelo Presolve = 0

In [11]:
# Optimización del modelo con configuración 2 con ambas instancias
modelo_presolve_I1 = optimizar_modelo_presolve(datos_1)
reporte_modelo(modelo_presolve_I1, "modelo Presolve = 0", "I1")


Set parameter Threads to value 1
Set parameter TimeLimit to value 600
Set parameter Presolve to value 0
¡Optimizando modelo con presolve = 0!

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-11800H @ 2.30GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 1 threads

Non-default parameters:
TimeLimit  600
Presolve  0
Threads  1

Optimize a model with 1428 rows, 1287 columns and 6300 nonzeros (Min)
Model fingerprint: 0xcbc56733
Model has 1188 linear objective coefficients
Variable types: 45 continuous, 1242 integer (1242 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+03]
  Objective range  [1e+00, 4e+01]
  Bounds range     [1e+00, 1e+03]
  RHS range        [1e+00, 1e+03]

Variable types: 45 continuous, 1242 integer (1242 binary)
Found heuristic solution: objective 741.0587894

Root relaxation: objective 2.239768e+02, 341 iterations, 0.01 s

In [12]:
modelo_presolve_I2 = optimizar_modelo_presolve(datos_2)
reporte_modelo(modelo_presolve_I2, "modelo Presolve = 0", "I2")

Set parameter Threads to value 1
Set parameter TimeLimit to value 600
Set parameter Presolve to value 0
¡Optimizando modelo con presolve = 0!

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-11800H @ 2.30GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 1 threads

Non-default parameters:
TimeLimit  600
Presolve  0
Threads  1

Optimize a model with 1506 rows, 1360 columns and 6669 nonzeros (Min)
Model fingerprint: 0xc86c9a53
Model has 1260 linear objective coefficients
Variable types: 46 continuous, 1314 integer (1314 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+02]
  Objective range  [7e+00, 6e+01]
  Bounds range     [1e+00, 2e+02]
  RHS range        [1e+00, 2e+02]

Variable types: 46 continuous, 1314 integer (1314 binary)

Root relaxation: objective 5.394012e+02, 455 iterations, 0.02 seconds (0.01 work units)

    Nodes    |    Curr

### Modelo MIPFocus = 1

In [13]:
modelo_mipfocus1_I1 = optimizar_modelo_mipfocus_1(datos_1)
reporte_modelo(modelo_mipfocus1_I1, "modelo MIPFocus = 1", "I1")

Set parameter Threads to value 1
Set parameter TimeLimit to value 600
Set parameter MIPFocus to value 1
¡Optimizando modelo con MIPFocus = 1!

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-11800H @ 2.30GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 1 threads

Non-default parameters:
TimeLimit  600
MIPFocus  1
Threads  1

Optimize a model with 1428 rows, 1287 columns and 6300 nonzeros (Min)
Model fingerprint: 0xcbc56733
Model has 1188 linear objective coefficients
Variable types: 45 continuous, 1242 integer (1242 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+03]
  Objective range  [1e+00, 4e+01]
  Bounds range     [1e+00, 1e+03]
  RHS range        [1e+00, 1e+03]

Presolve removed 783 rows and 397 columns
Presolve time: 0.03s
Presolved: 645 rows, 890 columns, 3102 nonzeros
Variable types: 32 continuous, 858 integer (858 binary)

Ro

In [27]:
modelo_mipfocus1_I2 = optimizar_modelo_mipfocus_1(datos_2)
reporte_modelo(modelo_mipfocus1_I2, "modelo MIPFocus = 1", "I2")

Set parameter Threads to value 1
Set parameter TimeLimit to value 600
Set parameter MIPFocus to value 1
¡Optimizando modelo con MIPFocus = 1!

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-11800H @ 2.30GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 1 threads

Non-default parameters:
TimeLimit  600
MIPFocus  1
Threads  1

Optimize a model with 1506 rows, 1360 columns and 6669 nonzeros (Min)
Model fingerprint: 0xc86c9a53
Model has 1260 linear objective coefficients
Variable types: 46 continuous, 1314 integer (1314 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+02]
  Objective range  [7e+00, 6e+01]
  Bounds range     [1e+00, 2e+02]
  RHS range        [1e+00, 2e+02]

Presolve removed 312 rows and 147 columns
Presolve time: 0.02s
Presolved: 1194 rows, 1213 columns, 5331 nonzeros
Variable types: 34 continuous, 1179 integer (1179 binary)

### Modelo MIPFocus = 3

In [15]:
modelo_mipfocus3_I1 = optimizar_modelo_mipfocus_3(datos_1)
reporte_modelo(modelo_mipfocus3_I1, "modelo MIPFocus = 3", "I1")

Set parameter Threads to value 1
Set parameter TimeLimit to value 600
Set parameter MIPFocus to value 3
¡Optimizando modelo con MIPFocus = 3!

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-11800H @ 2.30GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 1 threads

Non-default parameters:
TimeLimit  600
MIPFocus  3
Threads  1

Optimize a model with 1428 rows, 1287 columns and 6300 nonzeros (Min)
Model fingerprint: 0xcbc56733
Model has 1188 linear objective coefficients
Variable types: 45 continuous, 1242 integer (1242 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+03]
  Objective range  [1e+00, 4e+01]
  Bounds range     [1e+00, 1e+03]
  RHS range        [1e+00, 1e+03]

Presolve removed 783 rows and 397 columns
Presolve time: 0.03s
Presolved: 645 rows, 890 columns, 3102 nonzeros
Variable types: 32 continuous, 858 integer (858 binary)
Roo

In [16]:
modelo_mipfocus3_I2 = optimizar_modelo_mipfocus_3(datos_2)
reporte_modelo(modelo_mipfocus3_I2, "modelo MIPFocus = 3", "I2")

Set parameter Threads to value 1
Set parameter TimeLimit to value 600
Set parameter MIPFocus to value 3
¡Optimizando modelo con MIPFocus = 3!

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-11800H @ 2.30GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 1 threads

Non-default parameters:
TimeLimit  600
MIPFocus  3
Threads  1

Optimize a model with 1506 rows, 1360 columns and 6669 nonzeros (Min)
Model fingerprint: 0xc86c9a53
Model has 1260 linear objective coefficients
Variable types: 46 continuous, 1314 integer (1314 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+02]
  Objective range  [7e+00, 6e+01]
  Bounds range     [1e+00, 2e+02]
  RHS range        [1e+00, 2e+02]

Presolve removed 312 rows and 147 columns
Presolve time: 0.09s
Presolved: 1194 rows, 1213 columns, 5331 nonzeros
Variable types: 34 continuous, 1179 integer (1179 binary)

### Modelo Heuristics = 0

In [17]:
modelo_heuristics_I1 = optimizar_modelo_heuristics(datos_1)
reporte_modelo(modelo_heuristics_I1, "modelo Heuristics = 0", "I1")

Set parameter Threads to value 1
Set parameter TimeLimit to value 600
Set parameter Heuristics to value 0
¡Optimizando modelo con heuristics = 0!

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-11800H @ 2.30GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 1 threads

Non-default parameters:
TimeLimit  600
Heuristics  0
Threads  1

Optimize a model with 1428 rows, 1287 columns and 6300 nonzeros (Min)
Model fingerprint: 0xcbc56733
Model has 1188 linear objective coefficients
Variable types: 45 continuous, 1242 integer (1242 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+03]
  Objective range  [1e+00, 4e+01]
  Bounds range     [1e+00, 1e+03]
  RHS range        [1e+00, 1e+03]

Presolve removed 783 rows and 397 columns
Presolve time: 0.02s
Presolved: 645 rows, 890 columns, 3102 nonzeros
Variable types: 32 continuous, 858 integer (858 binar

In [18]:
modelo_heuristics_I2 = optimizar_modelo_heuristics(datos_2)
reporte_modelo(modelo_heuristics_I2, "modelo Heuristics = 0", "I2")

Set parameter Threads to value 1
Set parameter TimeLimit to value 600
Set parameter Heuristics to value 0
¡Optimizando modelo con heuristics = 0!

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-11800H @ 2.30GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 1 threads

Non-default parameters:
TimeLimit  600
Heuristics  0
Threads  1

Optimize a model with 1506 rows, 1360 columns and 6669 nonzeros (Min)
Model fingerprint: 0xc86c9a53
Model has 1260 linear objective coefficients
Variable types: 46 continuous, 1314 integer (1314 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+02]
  Objective range  [7e+00, 6e+01]
  Bounds range     [1e+00, 2e+02]
  RHS range        [1e+00, 2e+02]

Presolve removed 312 rows and 147 columns
Presolve time: 0.08s
Presolved: 1194 rows, 1213 columns, 5331 nonzeros
Variable types: 34 continuous, 1179 integer (1179 b

### Modelo tuneado

In [19]:
modelo_tuneado_I1 = optimizar_modelo_tuneado(datos_1, "I1")
reporte_modelo(modelo_tuneado_I1, "modelo tuneado", "I1")

Set parameter Threads to value 1
Set parameter TimeLimit to value 600
Set parameter TuneTimeLimit to value 3600
¡Optimizando parametros!

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-11800H @ 2.30GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 1 threads

Non-default parameters:
TimeLimit  600
Threads  1
TuneTimeLimit  3600

Start tuning.
Tune 34 parameters.

Solving model using baseline parameter set with TimeLimit=600s

Total tuning time limit is 3600s
-------------------------------------------------------------------------------

Testing candidate parameter set 1...

	Threads 1 (fixed)

Solving ConVRP_SLA_Reducido (Min) with random seed #1 ... MIP gap 0.01%
Solving ConVRP_SLA_Reducido (Min) with random seed #2 ... MIP gap 0.01%
Solving ConVRP_SLA_Reducido (Min) with random seed #3 ... MIP gap 0.01%

Summary candidate parameter set 1 
 # Name 

In [20]:
modelo_tuneado_I2 = optimizar_modelo_tuneado(datos_2, "I2")
reporte_modelo(modelo_tuneado_I2, "modelo tuneado", "I2")

Set parameter Threads to value 1
Set parameter TimeLimit to value 600
Set parameter TuneTimeLimit to value 3600
¡Optimizando parametros!

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-11800H @ 2.30GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 1 threads

Non-default parameters:
TimeLimit  600
Threads  1
TuneTimeLimit  3600

Start tuning.
Tune 34 parameters.

Solving model using baseline parameter set with TimeLimit=600s

Total tuning time limit is 3600s
-------------------------------------------------------------------------------

Testing candidate parameter set 1...

	Threads 1 (fixed)

Solving ConVRP_SLA_Reducido (Min) with random seed #1 ... MIP gap 0.01%
Solving ConVRP_SLA_Reducido (Min) with random seed #2 ... MIP gap 0.01%
Solving ConVRP_SLA_Reducido (Min) with random seed #3 ... MIP gap 0.01%

Summary candidate parameter set 1 
 # Name 

# Comparativa de modelos

In [21]:
modelos_I1 = [modelo_base_I1, modelo_cuts_I1, modelo_presolve_I1, modelo_mipfocus1_I1, modelo_mipfocus3_I1, modelo_heuristics_I1, modelo_tuneado_I1]

nombres = ["Baseline", "Cuts=0", "Presolve=0", "MIPF=1", "MIPF=3", "Heuristics=0","Tuneado"]

comparacion_modelos(modelos_I1, nombres, "I1")

=== Comparativa de modelos I1 ===



,Modelo,Tiempo,Valor Objetivo,Gap Final
0,Baseline,2.468,411.798227,0.0
1,Cuts=0,4.473,411.798227,0.0
2,Presolve=0,14.293,411.798227,0.0
3,MIPF=1,10.529,411.798227,0.0
4,MIPF=3,8.900,411.798227,0.0
5,Heuristics=0,3.346,411.798227,0.0
6,Tuneado,3.702,411.798227,0.0


In [28]:
modelos_I2 = [modelo_base_I2, modelo_cuts_I2, modelo_presolve_I2, modelo_mipfocus1_I2, modelo_mipfocus3_I2, modelo_heuristics_I2, modelo_tuneado_I2]

comparacion_modelos(modelos_I2, nombres, "I2")

=== Comparativa de modelos I2 ===



,Modelo,Tiempo,Valor Objetivo,Gap Final
0,Baseline,201.689,769.241931,0.000089
1,Cuts=0,600.006,769.241931,0.066428
2,Presolve=0,600.022,769.241931,0.042903
3,MIPF=1,462.658,769.241931,0.000087
4,MIPF=3,223.766,769.241931,0.000089
5,Heuristics=0,132.552,769.241931,0.000100
6,Tuneado,106.450,769.241931,0.000089


#  Analisis de sensibilidad

In [25]:
res_sens_I1 = analisis_sensibilidad_capacidad(modelo_tuneado_I1, "I1")

Set parameter Presolve to value 0
Set parameter Method to value 1
Set parameter Threads to value 1
=== Sensibilidad de Q, instancia I1 ===
Objetivo MIP: 411.7982 | Objetivo LP fijado: 411.7982
Restricciones de capacidad: 9 | activas: 0
Carga máxima observada: 130.00 (Q = 200.00)
Filas con precio sombra distinto de 0: 0


,Restricción,Q (RHS),Carga,Holgura,Activa,Precio sombra (Pi),SARHSLow,SARHSUp
2,Capacidad (R1421),200.0,130.0,70.0,False,0.0,130.0,inf
0,Capacidad (R1419),200.0,120.0,80.0,False,0.0,120.0,inf
1,Capacidad (R1420),200.0,120.0,80.0,False,0.0,120.0,inf
3,Capacidad (R1422),200.0,70.0,130.0,False,0.0,70.0,inf
4,Capacidad (R1423),200.0,70.0,130.0,False,0.0,70.0,inf
5,Capacidad (R1424),200.0,60.0,140.0,False,0.0,60.0,inf
6,Capacidad (R1425),200.0,0.0,200.0,False,0.0,0.0,inf
7,Capacidad (R1426),200.0,0.0,200.0,False,0.0,0.0,inf
8,Capacidad (R1427),200.0,0.0,200.0,False,0.0,0.0,inf


In [26]:
res_sens_I2 = analisis_sensibilidad_capacidad(modelo_tuneado_I2, "I2")

Set parameter Presolve to value 0
Set parameter Method to value 1
Set parameter Threads to value 1
=== Sensibilidad de Q, instancia I2 ===
Objetivo MIP: 769.2419 | Objetivo LP fijado: 769.2419
Restricciones de capacidad: 9 | activas: 0
Carga máxima observada: 89.00 (Q = 200.00)
Filas con precio sombra distinto de 0: 0


,Restricción,Q (RHS),Carga,Holgura,Activa,Precio sombra (Pi),SARHSLow,SARHSUp
8,Capacidad (R1505),200.0,89.0,111.0,False,0.0,89.0,inf
4,Capacidad (R1501),200.0,82.0,118.0,False,0.0,82.0,inf
3,Capacidad (R1500),200.0,75.0,125.0,False,0.0,75.0,inf
6,Capacidad (R1503),200.0,59.0,141.0,False,0.0,59.0,inf
7,Capacidad (R1504),200.0,57.0,143.0,False,0.0,57.0,inf
5,Capacidad (R1502),200.0,42.0,158.0,False,0.0,42.0,inf
0,Capacidad (R1497),200.0,26.0,174.0,False,0.0,26.0,inf
1,Capacidad (R1498),200.0,26.0,174.0,False,0.0,26.0,inf
2,Capacidad (R1499),200.0,0.0,200.0,False,0.0,0.0,inf
